In [ ]:
# === Micromotion 数值分析（展示层）===
# 调用 motion_analysis 库（phase-folding + FFT 低通调制深度拟合），逐离子测量
# RF micromotion 幅度 β(t) 与有效调制深度 q_eff，并与 trap_stability 的理论 q 交叉验证。
#
# 前置：先用 --continuous-sampling 采集满足 Nyquist 的数据，例如
#   python main.py --N 3 --time 20 --continuous-sampling \
#       --continuous-sampling-frames 20000 --interval 0.08 --step 10
#
# 物理：Paul 阱中离子运动为乘性调制 x(t) ≈ X_sec(t)·[1+(q/2)cos(Ωt)]，
# micromotion 幅度正比于瞬时 secular 位移（非恒幅加性模型）。

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from motion_analysis import (
    analyze_run,
    cross_check_q,
    plot_beta_vs_secular,
    plot_ion_timeseries,
    plot_qeff_histogram,
    plot_qeff_vs_displacement,
)

_here = Path.cwd()

# === 用户配置 ===
SAMPLE_RUN_DIR = None  # 目录名(如 't030.00_interval0.08_step10')或 None 自动选取第一个
CSV = "monolithic20241118.csv"   # data/ 下文件名或完整路径
CONFIG = "default.json"          # configs/ 下文件名或完整路径
SPECIES = "Ba135+"
AXES = ("x", "y", "z")


def _repo_root() -> Path:
    p = _here
    for _ in range(4):
        if (p / "data").is_dir() or (p / "continuous_sampling").is_dir():
            return p
        p = p.parent
    return _here.parent


def _resolve(p: str, sub: str) -> Path:
    pp = Path(p)
    if pp.is_absolute() or "/" in p:
        return pp
    return _repo_root() / sub / p


_root = _repo_root()
run_dir = (
    Path(SAMPLE_RUN_DIR)
    if SAMPLE_RUN_DIR
    else sorted(_root.glob("continuous_sampling/t0*_interval*_step*"))[0]
)
csv_path = _resolve(CSV, "data")
config_path = _resolve(CONFIG, "FieldConfiguration/configs")
print("run_dir =", run_dir)
print("csv     =", csv_path)
print("config  =", config_path)

In [ ]:
report = analyze_run(
    run_dir, csv_path=csv_path, config_path=config_path,
    species=SPECIES, axes=AXES,
)
traj = report.trajectory
print(f"ions={traj.n_ions}  dt={traj.dt_us*1e3:.2f} ns  RF={traj.freq_RF_MHz:.3f} MHz")
_ai = {"x": 0, "y": 1, "z": 2}
for ax in AXES:
    col = report.q_eff[:, _ai[ax]]
    col = col[np.isfinite(col)]
    print(f"  axis {ax}: q_eff median={np.median(col):.5f}  "
          f"(min={col.min():.5f}, max={col.max():.5f})")

cross = cross_check_q(
    report, csv_path=csv_path, config_path=config_path, species=SPECIES,
)
print("\nCross-check  measured_median / q_theory:")
for ax in AXES:
    print(f"  {ax}: theory={cross.q_theory[ax]:.5f}  "
          f"measured={cross.q_measured_median[ax]:.5f}  "
          f"ratio={cross.ratio[ax]:.3f}")
print(f"  center=({cross.center_um[0]:.2f}, {cross.center_um[1]:.2f}, {cross.center_um[2]:.2f}) um")

In [ ]:
# 选 q_eff 最大的轴与离子展示时序（含 secular 包络与 micromotion 残差）
best_ax = max(AXES, key=lambda a: abs(np.nanmedian(report.q_eff[:, _ai[a]])))
i0 = int(np.nanargmax(np.nan_to_num(report.q_eff[:, _ai[best_ax]])))
res = report.results[(i0, best_ax)]
r_axis = traj.r_um[:, i0, _ai[best_ax]]
plot_ion_timeseries(
    res, traj.t_us, r_axis,
    axis_label=best_ax, ion_idx=i0, freq_rf_MHz=traj.freq_RF_MHz,
)
plt.show()

In [ ]:
# q_eff 分布 / q_eff vs 偏离 RF null / β vs secular 位移
plot_qeff_histogram(report)
plt.show()
plot_qeff_vs_displacement(report, cross)
plt.show()
plot_beta_vs_secular(report, cross)
plt.show()

## 解读

- **q_eff 直方**：RF 径向轴（被 RF 驱动）q_eff 显著非零；轴向（无 RF）q_eff≈0。
- **q_eff vs 偏离 RF null**：线性阱下 q_eff 应 ≈ q_theory 且与位置无关；离子偏离阱中心（晶格库仑排斥）时 q_eff 增大 → excess micromotion。
- **β vs secular 位移**：散点应沿斜率 q_theory/2 的直线，验证乘性调制模型。
- **ratio**：冷单粒子在阱中心 ratio ≈ 1；晶格中 ratio > 1 属预期 excess micromotion。